# Проверка API сомелье

Запустите сервис через `docker compose up --build -d` из корня проекта и выполняйте ячейки по порядку. Ключ OpenRouter в notebook не нужен. Четыре сценария вызывают реальный OpenRouter: всего 9 генераций. После проверки остановите сервис командой `docker compose stop sommelier`.


In [1]:
import secrets
import time
import uuid

import httpx

In [2]:
BASE_URL = "http://127.0.0.1:8805"

In [3]:
client = httpx.Client(base_url=BASE_URL, timeout=90.0, trust_env=False)


def show(response: httpx.Response):
    print(f"{response.request.method} {response.request.url} → {response.status_code}")
    if response.content:
        payload = response.json()
        display(payload)
    else:
        payload = None
    response.raise_for_status()
    return payload

In [4]:
show(client.get("/health"))

GET http://127.0.0.1:8805/health → 200


{'status': 'ok'}

{'status': 'ok'}

## Четыре независимые сессии

Карточки сокращены из `../vinishko/data/catalog/catalog.csv`. Каждый сценарий создаёт новый UUIDv7, открывает сессию, задаёт вопросы, проверяет историю и удаляет свою сессию. Красное Каберне проверяет вопросы по двум переданным кандидатам; остальные сценарии используют другие типы вина. Ответы модели выводятся для ручной оценки.


In [5]:
def new_uuid7() -> uuid.UUID:
    # Python 3.13 ещё не предоставляет uuid.uuid7().
    timestamp_ms = time.time_ns() // 1_000_000
    random_bits = secrets.randbits(74)
    value = (timestamp_ms & ((1 << 48) - 1)) << 80
    value |= 7 << 76
    value |= ((random_bits >> 62) & 0xFFF) << 64
    value |= 0b10 << 62
    value |= random_bits & ((1 << 62) - 1)
    return uuid.UUID(int=value)

In [6]:
def run_case(title: str, wine: dict, candidates: list[dict], questions: list[str]) -> dict:
    session_id = str(new_uuid7())
    path = f"/v1/sessions/{session_id}"
    opened = False
    print(f"\n=== {title} | session_id={session_id} ===")
    if candidates:
        print("Кандидаты:", [card["Название вина"] for card in candidates])
    try:
        response = client.post(path, json={"wine": wine, "candidates": candidates})
        opened = response.status_code == 201
        opening = show(response)
        assert opening["session_id"] == session_id

        for question in questions:
            print("Вопрос:", question)
            reply = show(client.post(f"{path}/messages", json={"content": question}))
            assert reply["session_id"] == session_id

        history = show(client.get(path))
        assert history["session_id"] == session_id
        assert [message["role"] for message in history["messages"]] == [
            "assistant", *[role for _ in questions for role in ("user", "assistant")]
        ]
        assert [message["content"] for message in history["messages"] if message["role"] == "user"] == questions
        print("Сценарий пройден:", title)
        return history
    finally:
        if opened:
            show(client.delete(path))

### 1. Красное Каберне: сравнение с кандидатами

Два вопроса явно называют карточки из `candidates`: Пино Нуар и Саперави.

In [7]:
cabernet_wine = {
    "Название вина": "100 оттенков красного. Каберне",
    "Категория": "Красное",
    "Цвет": "Темно-рубиновый",
    "Регион": "Кубань",
    "Сорт винограда": "Каберне Совиньон",
    "Описание": "В аромате спелая вишня, чёрная смородина, лёгкая пряность. Вкус плотный, гармоничный, с мягкими танинами и ягодным послевкусием.",
    "Винодельня": "Фанагория",
    "Slug": "fanagoriya-100-ottenkov-krasnogo-kaberne-kaberne-sovinon-krasnoe-suhoe-135",
    "Крепость, %": 13.5,
    "Сахар": "сухое",
}
cabernet_candidates = [
    {
        "Название вина": "100 оттенков красного. Пино Нуар",
        "Категория": "Красное",
        "Регион": "Кубань",
        "Сорт винограда": "Пино Нуар",
        "Описание": "Стройный развитый аромат с нотами брусники, вяленых лесных ягод и граната. Вкус лёгкий, многослойный и свежий.",
        "Винодельня": "Фанагория",
        "Slug": "fanagoriya-100-ottenkov-krasnogo-pino-nuar-krasnoe-suhoe-135",
        "Сахар": "сухое",
    },
    {
        "Название вина": "100 оттенков красного. Саперави",
        "Категория": "Красное",
        "Регион": "Кубань",
        "Сорт винограда": "Саперави",
        "Описание": "В аромате ежевика, черника, тёмная вишня и специи. Вкус насыщенный, с бархатными танинами.",
        "Винодельня": "Фанагория",
        "Slug": "fanagoriya-100-ottenkov-krasnogo-saperavi-krasnoe-suhoe-14",
        "Сахар": "сухое",
    },
]
cabernet_history = run_case(
    "Каберне и кандидаты",
    cabernet_wine,
    cabernet_candidates,
    [
        "Чем кандидат «100 оттенков красного. Пино Нуар» отличается от найденного Каберне?",
        "Если хочется более насыщенное вино, подойдёт ли кандидат «100 оттенков красного. Саперави»?",
    ],
)


=== Каберне и кандидаты | session_id=01a0dd32-f6e5-7d07-b4eb-cdf452033fcf ===
Кандидаты: ['100 оттенков красного. Пино Нуар', '100 оттенков красного. Саперави']
POST http://127.0.0.1:8805/v1/sessions/01a0dd32-f6e5-7d07-b4eb-cdf452033fcf → 201


{'session_id': '01a0dd32-f6e5-7d07-b4eb-cdf452033fcf',
 'message': {'role': 'assistant',
  'content': 'Перед вами красное сухое вино «100 оттенков красного. Каберне» от «Фанагории» с Кубани. Винтаж в карточке не указан, а крепость — 13,5%. По описанию это плотный, гармоничный Каберне Совиньон с мягкими танинами и ароматами спелой вишни, чёрной смородины и лёгкой пряности. Попробуйте подать его слегка прохладным, около 16–18 °C, в достаточно широком бокале, а аэрацию или декантацию добавлять только если после первой пробы покажется, что вину тесно.\nЧем я могу вам помочь?',
  'suggestions': ['Какие ещё ноты можно найти во вкусе?',
   'Как лучше подать это вино?']}}

Вопрос: Чем кандидат «100 оттенков красного. Пино Нуар» отличается от найденного Каберне?
POST http://127.0.0.1:8805/v1/sessions/01a0dd32-f6e5-7d07-b4eb-cdf452033fcf/messages → 201


{'session_id': '01a0dd32-f6e5-7d07-b4eb-cdf452033fcf',
 'message': {'role': 'assistant',
  'content': 'Оба вина — красные сухие из Кубани от «Фанагории», но это разные сорта и стили. Найденное «Каберне» по карточке плотное, с мягкими танинами, ароматами спелой вишни, чёрной смородины и лёгкой пряности. Кандидат «Пино Нуар» описан как более лёгкое, свежее и многослойное, с нотами брусники, вяленых лесных ягод и граната. То есть если хочется больше тела и ягодной плотности — скорее подойдёт найденное Каберне, а если нужен более стройный и свежий профиль — Пино Нуар. Крепость и цвет Пино Нуара в карточке не указаны, поэтому по этим параметрам сравнить нельзя.',
  'suggestions': None}}

Вопрос: Если хочется более насыщенное вино, подойдёт ли кандидат «100 оттенков красного. Саперави»?
POST http://127.0.0.1:8805/v1/sessions/01a0dd32-f6e5-7d07-b4eb-cdf452033fcf/messages → 201


{'session_id': '01a0dd32-f6e5-7d07-b4eb-cdf452033fcf',
 'message': {'role': 'assistant',
  'content': 'Скорее да: по описанию «Саперави» из этой же линейки выглядит более насыщенным — с ежевикой, черникой, тёмной вишней и специями, а вкус указан как насыщенный, с бархатными танинами. Найденное «Каберне» тоже плотное, но в карточке его танины названы мягкими, а аромат — более ягодно-вишнёвым и с лёгкой пряностью. Так что если хочется сместить акцент в сторону большей плотности и тёмных ягод, «Саперави» — логичный выбор; если важнее мягкость и гармоничность, остаётся найденное «Каберне». Крепость, цвет и винтаж у «Саперави» в карточке не указаны, поэтому по этим параметрам сравнить нельзя.',
  'suggestions': None}}

GET http://127.0.0.1:8805/v1/sessions/01a0dd32-f6e5-7d07-b4eb-cdf452033fcf → 200


{'session_id': '01a0dd32-f6e5-7d07-b4eb-cdf452033fcf',
 'messages': [{'role': 'assistant',
   'content': 'Перед вами красное сухое вино «100 оттенков красного. Каберне» от «Фанагории» с Кубани. Винтаж в карточке не указан, а крепость — 13,5%. По описанию это плотный, гармоничный Каберне Совиньон с мягкими танинами и ароматами спелой вишни, чёрной смородины и лёгкой пряности. Попробуйте подать его слегка прохладным, около 16–18 °C, в достаточно широком бокале, а аэрацию или декантацию добавлять только если после первой пробы покажется, что вину тесно.\nЧем я могу вам помочь?',
   'suggestions': ['Какие ещё ноты можно найти во вкусе?',
    'Как лучше подать это вино?']},
  {'role': 'user',
   'content': 'Чем кандидат «100 оттенков красного. Пино Нуар» отличается от найденного Каберне?',
   'suggestions': None},
  {'role': 'assistant',
   'content': 'Оба вина — красные сухие из Кубани от «Фанагории», но это разные сорта и стили. Найденное «Каберне» по карточке плотное, с мягкими танинами,

Сценарий пройден: Каберне и кандидаты
DELETE http://127.0.0.1:8805/v1/sessions/01a0dd32-f6e5-7d07-b4eb-cdf452033fcf → 204


### 2. Белый Рислинг: гастросочетание

In [8]:
riesling_history = run_case(
    "Adagum Riesling",
    {
        "Название вина": "Adagum Riesling",
        "Категория": "Белое",
        "Регион": "Кубань",
        "Сорт винограда": "Рислинг",
        "Описание": "Аромат белых цветов, яблок, груши и цитруса. Освежающий вкус с живой кислотностью и фруктово-цветочным послевкусием.",
        "Винодельня": "Olymp Winery",
        "Slug": "olymp-winery-adagum-riesling-risling-beloe-suhoe-11",
        "Крепость, %": 11,
        "Сахар": "сухое",
    },
    [],
    ["Подойдёт ли это вино к морской рыбе и почему?"],
)


=== Adagum Riesling | session_id=01a0dd33-bc09-7840-ad49-c14dc3c5d295 ===
POST http://127.0.0.1:8805/v1/sessions/01a0dd33-bc09-7840-ad49-c14dc3c5d295 → 201


{'session_id': '01a0dd33-bc09-7840-ad49-c14dc3c5d295',
 'message': {'role': 'assistant',
  'content': 'Найдено вино Adagum Riesling от винодельни Olymp Winery — белое сухое, винтаж в карточке не указан. По описанию это освежающий стиль с живой кислотностью и ароматами белых цветов, яблока и цитруса. Сорт Рислинг обычно даёт высокую кислотность и точный аромат, что согласуется с описанием. Подайте вино прохладным, но не ледяным, в бокале с умеренной чашей — так аромат раскроется, а кислотность не будет казаться резкой.\nЧем я могу вам помочь?',
  'suggestions': ['Какие ароматы здесь главные?',
   'Как лучше подавать это вино?']}}

Вопрос: Подойдёт ли это вино к морской рыбе и почему?
POST http://127.0.0.1:8805/v1/sessions/01a0dd33-bc09-7840-ad49-c14dc3c5d295/messages → 201


{'session_id': '01a0dd33-bc09-7840-ad49-c14dc3c5d295',
 'message': {'role': 'assistant',
  'content': 'Да, к морской рыбе это вино подойдёт — особенно если рыба приготовлена просто: на гриле, в духовке, на пару или с лёгким сливочно-лимонным соусом. Причина в сочетании сухости, живой кислотности и ароматов цитруса, яблока и белых цветов: кислотность рислинга освежает вкус после рыбы и хорошо снимает ощущение жирности, а 11% алкоголя не перегружают блюдо. С оговоркой: если рыба подаётся под сладким, очень пряным или тяжёлым томатным соусом, баланс может стать спорным — тогда лучше выбрать более нейтральное сухое белое.',
  'suggestions': None}}

GET http://127.0.0.1:8805/v1/sessions/01a0dd33-bc09-7840-ad49-c14dc3c5d295 → 200


{'session_id': '01a0dd33-bc09-7840-ad49-c14dc3c5d295',
 'messages': [{'role': 'assistant',
   'content': 'Найдено вино Adagum Riesling от винодельни Olymp Winery — белое сухое, винтаж в карточке не указан. По описанию это освежающий стиль с живой кислотностью и ароматами белых цветов, яблока и цитруса. Сорт Рислинг обычно даёт высокую кислотность и точный аромат, что согласуется с описанием. Подайте вино прохладным, но не ледяным, в бокале с умеренной чашей — так аромат раскроется, а кислотность не будет казаться резкой.\nЧем я могу вам помочь?',
   'suggestions': ['Какие ароматы здесь главные?',
    'Как лучше подавать это вино?']},
  {'role': 'user',
   'content': 'Подойдёт ли это вино к морской рыбе и почему?',
   'suggestions': None},
  {'role': 'assistant',
   'content': 'Да, к морской рыбе это вино подойдёт — особенно если рыба приготовлена просто: на гриле, в духовке, на пару или с лёгким сливочно-лимонным соусом. Причина в сочетании сухости, живой кислотности и ароматов цитруса

Сценарий пройден: Adagum Riesling
DELETE http://127.0.0.1:8805/v1/sessions/01a0dd33-bc09-7840-ad49-c14dc3c5d295 → 204


### 3. Розовое: подача к лёгкому блюду

In [9]:
rose_history = run_case(
    "Adagum Estate Rose",
    {
        "Название вина": "Adagum Estate Rose",
        "Категория": "Розовое",
        "Цвет": "Нежный бледно-лососевый",
        "Регион": "Кубань",
        "Сорт винограда": "Каберне Совиньон, Мерло",
        "Описание": "Свежий аромат клубники, граната и красной смородины. Вкус изящный, с приятной кислотностью и лёгким ягодным послевкусием.",
        "Винодельня": "Olymp Winery",
        "Slug": "olymp-winery-adagum-estate-rose-kaberne-sovinon-rozovoe-suhoe-11",
        "Крепость, %": 11,
        "Сахар": "сухое",
    },
    [],
    ["Как подать это розовое к салату с морепродуктами?"],
)


=== Adagum Estate Rose | session_id=01a0dd34-21dc-7cf2-8ba7-30b8a66d592d ===
POST http://127.0.0.1:8805/v1/sessions/01a0dd34-21dc-7cf2-8ba7-30b8a66d592d → 201


{'session_id': '01a0dd34-21dc-7cf2-8ba7-30b8a66d592d',
 'message': {'role': 'assistant',
  'content': 'Найденное вино — Adagum Estate Rose от Olymp Winery, розовое сухое; винтаж в карточке не указан. По описанию это свежий, изящный розовый стиль с клубникой, гранатом и красной смородиной. Во вкусе заявлены приятная кислотность и лёгкое ягодное послевкусие. Подайте его хорошо охлаждённым в бокале для белого вина, чтобы подчеркнуть свежесть.\nЧем я могу вам помочь?',
  'suggestions': ['Какие ягоды здесь главные?',
   'До какой температуры охладить?']}}

Вопрос: Как подать это розовое к салату с морепродуктами?
POST http://127.0.0.1:8805/v1/sessions/01a0dd34-21dc-7cf2-8ba7-30b8a66d592d/messages → 201


{'session_id': '01a0dd34-21dc-7cf2-8ba7-30b8a66d592d',
 'message': {'role': 'assistant',
  'content': 'Да, к салату с морепродуктами это розе подходит — особенно если заправка лёгкая: оливковое масло, лимон, травы, соль и перец. Подайте хорошо охлаждённым, ориентир для розового — 8–10 °C, в бокале для белого вина; открывайте перед едой, декантировать не нужно. Свежая кислотность и ягодный профиль — клубника, гранат, красная смородина — поддержат креветки, краба, гребешки, мидии и нежирную рыбу. С майонезом, сливочным соусом или сладкой заправкой — с оговоркой: вино может показаться резче, а сливочность приглушит фруктовость. Если салат острый или с копчёной рыбой, лучше другое, более нейтральное белое. По карточке игристость не определена; если при открытии окажутся пузырьки, охладите сильнее и открывайте осторожно.',
  'suggestions': None}}

GET http://127.0.0.1:8805/v1/sessions/01a0dd34-21dc-7cf2-8ba7-30b8a66d592d → 200


{'session_id': '01a0dd34-21dc-7cf2-8ba7-30b8a66d592d',
 'messages': [{'role': 'assistant',
   'content': 'Найденное вино — Adagum Estate Rose от Olymp Winery, розовое сухое; винтаж в карточке не указан. По описанию это свежий, изящный розовый стиль с клубникой, гранатом и красной смородиной. Во вкусе заявлены приятная кислотность и лёгкое ягодное послевкусие. Подайте его хорошо охлаждённым в бокале для белого вина, чтобы подчеркнуть свежесть.\nЧем я могу вам помочь?',
   'suggestions': ['Какие ягоды здесь главные?',
    'До какой температуры охладить?']},
  {'role': 'user',
   'content': 'Как подать это розовое к салату с морепродуктами?',
   'suggestions': None},
  {'role': 'assistant',
   'content': 'Да, к салату с морепродуктами это розе подходит — особенно если заправка лёгкая: оливковое масло, лимон, травы, соль и перец. Подайте хорошо охлаждённым, ориентир для розового — 8–10 °C, в бокале для белого вина; открывайте перед едой, декантировать не нужно. Свежая кислотность и ягодный

Сценарий пройден: Adagum Estate Rose
DELETE http://127.0.0.1:8805/v1/sessions/01a0dd34-21dc-7cf2-8ba7-30b8a66d592d → 204


### 4. Игристый брют: температура подачи

In [10]:
brut_history = run_case(
    "Abrau-Durso Reserve Brut",
    {
        "Название вина": "Abrau-Durso Reserve Brut",
        "Категория": "Белое",
        "Регион": "Кубань",
        "Сорт винограда": "Пино Блан, Рислинг, Шардоне",
        "Описание": "Аромат зелёного яблока, груши и цветов акации. Структурный вкус с живой кислотностью, нотами яблока и цедры цитрусов.",
        "Винодельня": "Абрау-Дюрсо",
        "Slug": "abrau-dyurso-abrau-durso-reserve-brut-shardone-beloe-bryut-115",
        "Крепость, %": 11.5,
        "Сахар": "брют",
        "Игристое": "да",
    },
    [],
    ["До какой температуры охладить этот брют перед подачей?"],
)


=== Abrau-Durso Reserve Brut | session_id=01a0dd34-89ba-7224-9b22-b7f8fd789d01 ===
POST http://127.0.0.1:8805/v1/sessions/01a0dd34-89ba-7224-9b22-b7f8fd789d01 → 201


{'session_id': '01a0dd34-89ba-7224-9b22-b7f8fd789d01',
 'message': {'role': 'assistant',
  'content': 'Найденное вино — Abrau-Durso Reserve Brut от винодельни «Абрау-Дюрсо», белое игристое категории брют.\nВ аромате и вкусе у него читаются зелёное яблоко, груша, цветы акации и живая цитрусовая кислотность.\nЭто свежий, структурный игристый стиль с яблочно-цитрусовой ясностью.\nПодайте его хорошо охлаждённым в тюльпановидном бокале и открывайте бутылку без встряхивания, контролируя пробку.\nЧем я могу вам помочь?',
  'suggestions': ['Что главное во вкусе?', 'Как лучше охладить и открыть?']}}

Вопрос: До какой температуры охладить этот брют перед подачей?
POST http://127.0.0.1:8805/v1/sessions/01a0dd34-89ba-7224-9b22-b7f8fd789d01/messages → 201


{'session_id': '01a0dd34-89ba-7224-9b22-b7f8fd789d01',
 'message': {'role': 'assistant',
  'content': 'Точной температуры подачи в карточке нет, но для белого игристого брют обычно ориентируются на 6–10 °C. Если хотите подчеркнуть свежесть и цитрусовую кислотность, подавайте ближе к 6–8 °C; если хочется, чтобы ярче раскрылись яблоко, груша и цветочные ноты, — ближе к 9–10 °C. Сильно переохлаждать не стоит: аромат станет глуше. Открывайте бутылку без встряхивания, удерживая пробку, и наливайте в тюльпановидный бокал.',
  'suggestions': None}}

GET http://127.0.0.1:8805/v1/sessions/01a0dd34-89ba-7224-9b22-b7f8fd789d01 → 200


{'session_id': '01a0dd34-89ba-7224-9b22-b7f8fd789d01',
 'messages': [{'role': 'assistant',
   'content': 'Найденное вино — Abrau-Durso Reserve Brut от винодельни «Абрау-Дюрсо», белое игристое категории брют.\nВ аромате и вкусе у него читаются зелёное яблоко, груша, цветы акации и живая цитрусовая кислотность.\nЭто свежий, структурный игристый стиль с яблочно-цитрусовой ясностью.\nПодайте его хорошо охлаждённым в тюльпановидном бокале и открывайте бутылку без встряхивания, контролируя пробку.\nЧем я могу вам помочь?',
   'suggestions': ['Что главное во вкусе?', 'Как лучше охладить и открыть?']},
  {'role': 'user',
   'content': 'До какой температуры охладить этот брют перед подачей?',
   'suggestions': None},
  {'role': 'assistant',
   'content': 'Точной температуры подачи в карточке нет, но для белого игристого брют обычно ориентируются на 6–10 °C. Если хотите подчеркнуть свежесть и цитрусовую кислотность, подавайте ближе к 6–8 °C; если хочется, чтобы ярче раскрылись яблоко, груша и цв

Сценарий пройден: Abrau-Durso Reserve Brut
DELETE http://127.0.0.1:8805/v1/sessions/01a0dd34-89ba-7224-9b22-b7f8fd789d01 → 204


In [11]:
client.close()